# Final 784→1000→10 learning-rule run

This Colab runs **one learning rule** under the frozen final protocol,
including both sequential and interleaved conditions.

Run it once for each rule:

1. `backprop`
2. `feedback_alignment`
3. `predictive_coding`
4. `hebbian`

Download the ZIP produced by the last cell immediately. The second
notebook combines the four ZIP files into the final tables and plots.

The primary outcome is catastrophic forgetting. SNR and cosine are
secondary, layer-wise update diagnostics. Analysis probes never update
model weights.


## 1. Get the repository and install dependencies

This currently targets the `post-merge-fixes` branch containing the
standardized runner and analysis code.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

REPO_URL = "https://github.com/lauraneedham/Catastrophically-Forgotten.git"
REPO_BRANCH = "post-merge-fixes"
REPO_DIR = Path("/content/Catastrophically-Forgotten")

if (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin"])
    subprocess.check_call(
        ["git", "-C", str(REPO_DIR), "switch", REPO_BRANCH]
    )
    subprocess.check_call(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH]
    )
else:
    subprocess.check_call(
        [
            "git",
            "clone",
            "--branch",
            REPO_BRANCH,
            REPO_URL,
            str(REPO_DIR),
        ]
    )

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")]
)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("Repository:", REPO_DIR)
print(
    "Commit:",
    subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
        text=True,
    ).strip(),
)


## 2. Select exactly one rule

Do not change the architecture or common protocol. You may reduce
`max_probe_batches` only if PC probing is too slow; record the change.


In [ ]:
import torch

from src.data import download_mnist
from src.experiments.comparative import (
    ComparativeConfig,
    FINAL_RULES,
    run_rule_experiment,
    save_rule_experiment,
)

RULE_TO_RUN = "backprop"  # change to one FINAL_RULES value per session
COLLECT_UPDATE_METRICS = True

config = ComparativeConfig(
    num_inputs=784,
    num_hidden=1000,
    num_outputs=10,
    activation_type="sigmoid",
    bias=False,
    learning_rate=0.001,
    batch_size=32,
    phase1_recorded_epochs=20,
    phase2_recorded_epochs=20,
    record_initial_baseline=True,
    train_prop=0.8,
    keep_prop=1.0,
    data_split_seed=0,
    model_seed=0,
    data_order_seed=1000,
    max_probe_batches=8,
    phase2_analysis_epochs=(1, 5, 10, 20),
)
config.validate()
assert RULE_TO_RUN in FINAL_RULES
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Rule:", RULE_TO_RUN)
print("Device:", DEVICE)
print("Architecture:", config.architecture)


## 3. Load the single shared MNIST split

The private split seed makes the train/validation indices independent
of model initialization and previous random-number use.


In [ ]:
train_set, valid_set, test_set = download_mnist(
    train_prop=config.train_prop,
    keep_prop=config.keep_prop,
    seed=config.data_split_seed,
)
assert len(train_set) + len(valid_set) == 60_000
assert len(test_set) == 10_000
print("Shared split verified.")


## 3.1 Fast integration smoke test

Keep this enabled the first time each learning rule is run. It uses the
final 1,000-unit architecture but only a small deterministic subset and
two recorded epochs per phase. It catches model/collector/dependency
problems before the long full-MNIST cell.


In [ ]:
from dataclasses import replace
from torch.utils.data import Subset

RUN_SMOKE_TEST = True
if RUN_SMOKE_TEST:
    smoke_train = Subset(
        train_set.dataset,
        list(train_set.indices[:512]),
    )
    smoke_valid = Subset(
        valid_set.dataset,
        list(valid_set.indices[:256]),
    )
    smoke_config = replace(
        config,
        phase1_recorded_epochs=2,
        phase2_recorded_epochs=2,
        max_probe_batches=1,
        phase2_analysis_epochs=(1, 2),
    )
    smoke_result = run_rule_experiment(
        RULE_TO_RUN,
        smoke_train,
        smoke_valid,
        smoke_config,
        device=DEVICE,
        collect_update_metrics=True,
        verbose=False,
    )
    print("Smoke test passed.")
    print(smoke_result["summaries"])


## 4. Run sequential and interleaved conditions

This is the long cell. It prints accuracy after every recorded epoch.
Both conditions restart from the same seed and private loader order.


In [ ]:
result = run_rule_experiment(
    RULE_TO_RUN,
    train_set,
    valid_set,
    config,
    device=DEVICE,
    collect_update_metrics=COLLECT_UPDATE_METRICS,
    verbose=True,
)
assert result["phase1_conditions_match"], (
    "Sequential and interleaved phase-one endpoints did not match."
)
print("Run complete.")


## 5. Inspect the performance summary

In [ ]:
import pandas as pd
from IPython.display import display

summary_df = pd.DataFrame(result["summaries"])
trace_df = pd.DataFrame(result["traces"])
update_df = pd.DataFrame(result["update_metrics"])
display(summary_df)

assert set(summary_df["condition"]) == {"sequential", "interleaved"}
assert (summary_df["architecture"] == "784-1000-10").all()
assert (summary_df["learning_rate"] == 0.001).all()
assert summary_df["phase1_old_accuracy"].min() >= 80.0, (
    "The rule did not pass the predefined phase-one competence gate."
)


In [ ]:
import matplotlib.pyplot as plt

trace_df["global_epoch"] = trace_df["recorded_epoch"]
trace_df.loc[trace_df["phase"] == 2, "global_epoch"] += (
    config.phase1_recorded_epochs
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for axis, condition in zip(axes, ("sequential", "interleaved")):
    subset = trace_df[trace_df["condition"] == condition]
    axis.plot(
        subset["global_epoch"],
        subset["old_accuracy"],
        label="Old digits 0-5",
        linewidth=2,
    )
    axis.plot(
        subset["global_epoch"],
        subset["new_accuracy"],
        label="New digits 6-9",
        linewidth=2,
    )
    axis.axvline(
        config.phase1_recorded_epochs + 0.5,
        color="black",
        linestyle="--",
        alpha=0.6,
    )
    axis.set_title(f"{RULE_TO_RUN}: {condition}")
    axis.set_xlabel("Recorded epoch")
    axis.set_ylim(-2, 102)
    axis.grid(alpha=0.25)
axes[0].set_ylabel("Validation accuracy (%)")
axes[1].legend()
plt.tight_layout()
plt.show()


## 6. Inspect update SNR and cosine

Values are kept separate by layer, checkpoint, probe split, and
condition. Do not average the hidden and output layers together.


In [ ]:
if COLLECT_UPDATE_METRICS:
    display(
        update_df[
            [
                "condition",
                "checkpoint",
                "probe_split",
                "layer",
                "snr",
                "cosine_mean",
                "cosine_sem",
                "cosine_valid_batches",
            ]
        ]
    )

    final_update = update_df[
        update_df["checkpoint"] == "phase2_epoch_20"
    ].copy()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for axis, metric, title in (
        (axes[0], "snr", "Update SNR"),
        (axes[1], "cosine_mean", "Cosine to BP descent"),
    ):
        labels = (
            final_update["condition"]
            + "\n"
            + final_update["probe_split"]
            + "\n"
            + final_update["layer"]
        )
        axis.bar(range(len(final_update)), final_update[metric])
        axis.set_xticks(range(len(final_update)))
        axis.set_xticklabels(labels, rotation=70, ha="right", fontsize=8)
        axis.set_title(title)
        axis.grid(axis="y", alpha=0.25)
    axes[1].axhline(0, color="black", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


## 7. Save and download this rule's artifact bundle

Keep the ZIP unchanged. The combining notebook expects its CSV/JSON
files and directory structure.


In [ ]:
import json
import platform
import shutil
from google.colab import files

OUTPUT_ROOT = Path("/content/final_rule_outputs")
rule_output_dir = OUTPUT_ROOT / RULE_TO_RUN
rule_output_dir.mkdir(parents=True, exist_ok=True)

result["environment"] = {
    "repository": REPO_URL,
    "branch": REPO_BRANCH,
    "commit": subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
        text=True,
    ).strip(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "device": DEVICE,
}
save_rule_experiment(result, rule_output_dir)
(rule_output_dir / "README.txt").write_text(
    "Final 784-1000-10 run. Keep all files together for the "
    "combining notebook.\n",
    encoding="utf-8",
)

zip_base = Path("/content") / f"final_784_1000_10_{RULE_TO_RUN}"
zip_path = Path(
    shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_ROOT, base_dir=RULE_TO_RUN)
)
print("Saved:", zip_path)
files.download(str(zip_path))
